# Associação entre variáveis categóricas

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_06/01_associacao_variaveis_categoricas.ipynb)

Uma tabela cruzada conta combinações de categorias. Para interpretar associação,
precisamos comparar proporções condicionais e explicitar qual variável define o
denominador. Contagens maiores podem resultar apenas de grupos maiores.

![Contagens observadas passam a proporções condicionais, valores esperados e uma medida de associação.](imagens/01_tabela_condicional.svg)

## 1. Das contagens às proporções condicionais

A primeira célula de análise lê `registros_associacao.csv`, cruza período e tema
educação e apresenta duas saídas: contagens observadas e porcentagens dentro de
cada período. Leia primeiro os totais e confirme o denominador; somente depois
compare as proporções.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_06'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dados = pd.read_csv("dados/registros_associacao.csv")
tabela = pd.crosstab(dados["periodo"], dados["tema_educacao"])
proporcoes_linha = pd.crosstab(
    dados["periodo"], dados["tema_educacao"], normalize="index"
)
display(tabela)
display((100 * proporcoes_linha).round(1))

## 2. Da independência esperada à discrepância observada

Sob independência, a frequência esperada de uma célula é o produto dos totais de
sua linha e coluna dividido pelo total geral:

$$E_{ij}=\frac{n_{i\cdot}n_{\cdot j}}{n}$$

A estatística qui-quadrado resume discrepâncias. O $V$ de Cramér a normaliza:

$$V=\sqrt{\frac{\chi^2}{n(k-1)}}$$

Aqui, $n$ é o número de documentos e $k$ é o menor número de categorias entre
linhas e colunas. $V$ varia de 0 a 1, mas nenhuma dessas medidas informa direção
causal ou importância histórica (Agresti, 2018).

In [ ]:
observada = tabela.to_numpy(dtype=float)
esperada = np.outer(observada.sum(axis=1), observada.sum(axis=0)) / observada.sum()
qui_quadrado = ((observada - esperada) ** 2 / esperada).sum()
n = observada.sum()
k = min(observada.shape)
v_cramer = np.sqrt(qui_quadrado / (n * (k - 1)))
display(pd.DataFrame(esperada, index=tabela.index, columns=tabela.columns).round(2))
print(f"Qui-quadrado descritivo: {qui_quadrado:.3f}")
print(f"V de Cramér: {v_cramer:.3f}")

(100 * proporcoes_linha["sim"]).plot.bar(color="#5a7d8c")
plt.ylabel("Documentos com tema educação (%)")
plt.xlabel("Período")
plt.title("Proporção condicionada pelo período — dados fictícios")
plt.xticks(rotation=0); plt.ylim(0, 100); plt.tight_layout(); plt.show()

A tabela e o gráfico descrevem este conjunto fictício. Antes de atribuir a
diferença ao tempo, seria necessário examinar gênero, local, seleção do corpus e
mudança na classificação. O $V$ resume intensidade no arranjo observado, não uma
escala universal de relevância.

## U06-A02 — Atividade integrada — tabela associativa documentada

1. Escolha outro par categórico (`local`, `genero` ou `tem_ilustracao`).
2. Produza contagens e proporções com denominador explícito.
3. Calcule frequências esperadas e $V$ de Cramér.
4. Inspecione ao menos quatro registros e escreva uma conclusão limitada.

**Resposta e interpretação:** Escreva aqui.

## Passagem

Categorias exigem tabelas condicionadas. Para quantidades, o próximo notebook
examina forma, ordem, extremos e estratificação antes de resumir a associação.

**Leitura:** Agresti (2018) e Cramér (1946), listados em `referencias.md`.